# 07 — Sistema híbrido CNN + landmarks de MediaPipe

**Qué hace:** descarga el modelo `hand_landmarker.task` de MediaPipe Tasks, extrae vectores de landmarks de mano para train/val/test y los guarda en CSV, entrena un `MLPClassifier` de scikit-learn sobre esos vectores, y arma un predictor híbrido: si MediaPipe detecta una mano usa el MLP sobre landmarks, si no, cae de vuelta a la CNN fine-tuneada (`finetunned.keras`). Evalúa el sistema híbrido completo y prueba con una imagen individual.

**Consume:** `asl_split_nuevo/train|test|validate/<clase>/*` (producido por `02_split_por_grupos.ipynb`), `finetunned.keras` (producido por `05_fine_tuning.ipynb`), `./test-out/wt.jpg` (imagen de prueba individual).

**Produce:** `hand_landmarker.task` (modelo descargado de MediaPipe), `asl_train_landmarks.csv`, `asl_val_landmarks.csv`, `asl_test_landmarks.csv` (vectores de landmarks por split).

## Descarga del modelo `hand_landmarker.task` de MediaPipe

In [4]:
import urllib.request

# Descargamos el modelo oficial de MediaPipe para manos
url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
urllib.request.urlretrieve(url, "hand_landmarker.task")
print("Modelo hand_landmarker.task descargado con éxito.")

Modelo hand_landmarker.task descargado con éxito.


## Configuración de MediaPipe Tasks (HandLandmarker)

In [25]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as python_mp
from mediapipe.tasks.python import vision
from pathlib import Path

base_options = python_mp.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands = 2,
    min_hand_detection_confidence = 0.3
)


## Vector de landmarks por mano (`hand_vector`) y extracción por split

In [24]:
FINGER_DISTANCE = [(4, 8), (8,12), (12,16), (16,20)]

def hand_vector(landmarks):
    coords = np.array([[landmark.x, landmark.y, landmark.z] for landmark in landmarks])
    coords -= coords[0]

    distance = [np.linalg.norm(coords[a] - coords[b]) for a,b in FINGER_DISTANCE]

    return np.concatenate([coords.flatten(), distance])

In [23]:
HAND_SIZE = 67
VECTOR_SIZE = HAND_SIZE*2 

def extract_split(dataset, landmarker):
    path = Path(dataset)
    rows, fouls = [], {}

    classes = sorted([directory.name for directory in dataset.iterdir() if directory.is_dir()])
    for class_ in classes:
        total, no_hand = 0, 0
        for path in (dataset / class_).glob('*.*'):
            img = cv2.imread(str(path))

            if img is None:
                continue
                
            total+=1
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

            result = landmarker.detect(mp_image)

            if not result.hand_landmarks:
                no_hand +=1
                continue

            vector = np.zeros(VECTOR_SIZE, dtype=np.float32)

            for i, hand in enumerate(result.hand_landmarks[:2]):
                vector[i*HAND_SIZE: (i+1)*HAND_SIZE] = hand_vector(hand)

            rows.append([class_]+vector.tolist())

        fouls[class_] = (no_hand, total)
        print(f"  {class_}: {total - no_hand}/{total} detectadas")

    columns = ['clase'] + [f'v{i}' for i in range(VECTOR_SIZE)]
    return pd.DataFrame(rows, columns=columns), fouls

## Extracción de landmarks para train/val/test y guardado en CSV

In [75]:
DESTINO = Path('./asl_split_nuevo')

with vision.HandLandmarker.create_from_options(options) as landmarker:
    df_train, rep_train = extract_split(DESTINO / 'train', landmarker)
    df_val, rep_val = extract_split(DESTINO / 'validate', landmarker)
    df_test, rep_test = extract_split(DESTINO / 'test', landmarker)

df_train.to_csv('asl_train_landmarks.csv', index=False)
df_val.to_csv('asl_val_landmarks.csv', index=False)
df_test.to_csv('asl_test_landmarks.csv', index=False)
print(f'Train: {len(df_train)}')

I0000 00:00:1783642090.700121 1494789 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1783642090.939758 1494789 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1783642091.004768 1494792 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783642091.027656 1494792 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783642091.096037 1494793 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  0: 555/796 detectadas
  1: 261/261 detectadas
  2: 213/213 detectadas
  3: 195/195 detectadas
  4: 263/263 detectadas
  5: 134/134 detectadas
  6: 206/206 detectadas
  7: 145/145 detectadas
  8: 305/305 detectadas
  9: 168/168 detectadas


E0000 00:00:1783642211.113254 1494790 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-09T18:24:11.106507-06:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


  A: 173/236 detectadas
  B: 241/241 detectadas
  C: 202/253 detectadas
  D: 261/261 detectadas
  E: 111/121 detectadas
  F: 224/224 detectadas
  G: 184/184 detectadas


E0000 00:00:1783642271.118230 1494790 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-09T18:24:11.106507-06:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


...

  V: 370/370 detectadas
  W: 312/312 detectadas
  X: 391/431 detectadas


E0000 00:00:1783643651.444853 1494790 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-09T18:39:11.414059-06:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


  Y: 393/395 detectadas
  Z: 574/574 detectadas


E0000 00:00:1783643686.063582 1494790 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-09T18:39:11.414059-06:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


Train: 7529


## Prueba rápida: cargar la CNN fine-tuneada y predecir una imagen individual

In [6]:
import tensorflow as tf

model_cargadox = tf.keras.models.load_model('finetunned.keras')


In [13]:
img = cv2.imread(str('./test-out/wt.jpg'))


h, w = img.shape[:2]
esc = 224 / max(h, w)
nh, nw = int(h*esc), int(w*esc)
lienzo = np.zeros((224, 224, 3), dtype=np.uint8)
top, left = (224-nh)//2, (224-nw)//2
lienzo[top:top+nh, left:left+nw] = cv2.resize(img, (nw, nh))
testing = cv2.cvtColor(lienzo, cv2.COLOR_BGR2RGB)[np.newaxis, ...].astype('float32')

In [14]:
import numpy as np
import tensorflow as tf

class_names =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

pred = model_cargadox.predict(testing, verbose=0)[0]



print(class_names[np.argmax(pred)])

L


## Preparación de datos de landmarks: `LabelEncoder` + `StandardScaler`

In [15]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [16]:
df_train = pd.read_csv('asl_train_landmarks.csv')
df_val = pd.read_csv('asl_val_landmarks.csv')
df_test = pd.read_csv('asl_test_landmarks.csv')

encoder = LabelEncoder()

y_train = encoder.fit_transform(df_train["clase"])
y_val = encoder.fit_transform(df_val["clase"])
y_test = encoder.fit_transform(df_test["clase"])

scaler = StandardScaler()

x_train = scaler.fit_transform(df_train.drop(columns='clase'))
x_val = scaler.fit_transform(df_val.drop(columns='clase'))
x_test = scaler.fit_transform(df_test.drop(columns='clase'))

clases = list(encoder.classes_) 

print(clases)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


## Entrenamiento del `MLPClassifier` sobre los landmarks

In [17]:
from sklearn.neural_network import MLPClassifier


classifier = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    alpha=1e-3,
    early_stopping=True,
    n_iter_no_change=5,
    max_iter=500,
    random_state=42
)

classifier.fit(x_train, y_train)
print("Val accuracy:", classifier.score(x_test, y_test))

Val accuracy: 0.8921266682460712


## Carga de la CNN y funciones del predictor híbrido

In [18]:
import tensorflow as tf

cnn = tf.keras.models.load_model('finetunned.keras')   # tu mejor CNN
# Ambos ordenan alfabético -> mismos índices. Blindaje de una línea:
assert list(encoder.classes_) == sorted(encoder.classes_), "Revisar orden de clases"
assert ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z'] == sorted(encoder.classes_), "Revisar orden de clases"

In [19]:
def preparar_para_cnn(img_bgr):
    """Replica el padding+224 que la CNN espera."""
    h, w = img_bgr.shape[:2]
    esc = 224 / max(h, w)
    nh, nw = int(h*esc), int(w*esc)
    lienzo = np.zeros((224, 224, 3), dtype=np.uint8)
    top, left = (224-nh)//2, (224-nw)//2
    lienzo[top:top+nh, left:left+nw] = cv2.resize(img_bgr, (nw, nh))
    return cv2.cvtColor(lienzo, cv2.COLOR_BGR2RGB)[np.newaxis, ...].astype('float32')

In [20]:
def handle_landmark(img, landmarker_):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    return landmarker_.detect(mp_image)

def handleVector(hand_landmarks):
    vector = np.zeros(VECTOR_SIZE, dtype=np.float32)
    for i, hand in enumerate(hand_landmarks[:2]):
        vector[i*HAND_SIZE: (i+1)*HAND_SIZE] = hand_vector(hand)

    vector_std = scaler.transform(vector.reshape(1, -1))   # sklearn: mismo scaler
    return classifier.predict_proba(vector_std)[0]
    

In [21]:
def predict(path, landmarker):
    img = cv2.imread(str(path))
    if img is None:
        return {"error": "ilegible"}
    
    landmark = handle_landmark(img, landmarker)

    if landmark.hand_landmarks:
        prediction = handleVector(landmark.hand_landmarks)
        via = 'landmarks'
    else:
        prediction =  cnn.predict(preparar_para_cnn(img), verbose=0)[0]
        via = 'cnn'

    return {"letra": clases[int(prediction.argmax())],
            "confianza": float(prediction.max()),
            "via": via}


## Evaluación del sistema híbrido completo

In [114]:
from mediapipe.tasks.python import vision

def evaluar(dir, landmarker):
    marker = {"landmarks": [0, 0], "cnn": [0, 0]}
    for class_dir in sorted(Path(dir).iterdir()):
        if not class_dir.is_dir():
            continue

        for image_path in class_dir.glob('*.*'): 
            response = predict(image_path, landmarker)
            if 'error' in response:
                continue

            marker[response["via"]][0] += int(response["letra"] == class_dir.name)
            marker[response["via"]][1] += 1

    tot_ok = sum(v[0] for v in marker.values())
    tot = sum(v[1] for v in marker.values())
    print(f"HÍBRIDO GLOBAL: {tot_ok/tot:.4f}")
    for via, (ok, n) in marker.items():
        if n: print(f"  {via}: {ok/n:.4f}  ({n/tot:.0%} del tráfico)")
        


In [119]:
with vision.HandLandmarker.create_from_options(options) as landmarker:
    evaluar(DESTINO / 'test', landmarker)

I0000 00:00:1783650446.293935 1716088 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1783650446.328844 1716095 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783650446.346253 1716093 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/denilsonbarahona/Documents/IA/mod

HÍBRIDO GLOBAL: 0.8996
  landmarks: 0.8921  (87% del tráfico)
  cnn: 0.9509  (13% del tráfico)


## Predicción sobre una imagen individual

In [30]:
with vision.HandLandmarker.create_from_options(options) as landmarker:
    resultado = predict('./test-out/wt.jpg', landmarker)

print(resultado)
# {'letra': 'D', 'confianza': 0.94, 'via': 'landmarks'}

I0000 00:00:1783705410.073485   60708 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1783705410.079980   60711 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783705410.085031   60716 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/denilsonbarahona/Documents/IA/model-sign-language/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'letra': 'T', 'confianza': 0.7179793130267681, 'via': 'landmarks'}
